# 数据对象：Tensor、Tile 与 Ptr

Pro 用三类对象描述第 1 章存储地图上的数据：**Tensor**（GM 中的数据视图）、**Tile**（片上 Buffer 中的数据块）、**Ptr**（裸指针）。本节在 02.02 参数体系的基础上，讲清三者的职责边界与构造方式——后面的地址规划、搬运与同步都建立在这三类对象之上。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 职责对照

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">对象</th><th style="text-align:left;">所在位置</th><th style="text-align:left;">职责</th><th style="text-align:left;">关键点</th></tr>
<tr><td><b>Tensor</b></td><td>GM</td><td>Kernel 的输入/输出数据视图</td><td>记录 dtype/shape/stride/layout，<b>不负责申请 GM 内存</b>（内存由 Host 侧 PyTorch 分配）</td></tr>
<tr><td><b>Tile</b></td><td>片上 Buffer</td><td>计算与搬运接口的基本操作对象</td><td>记录 shape/dtype/内存空间/排布/绑定地址</td></tr>
<tr><td><b>Ptr</b></td><td>GM 地址</td><td>裸指针，配合 TilingData 构造动态视图</td><td>通过 make_tensor 包装后使用</td></tr>
</table>

一句话记忆：**Host 用 PyTorch 准备 GM 数据 → Kernel 用 Tensor"看"它 → 搬到片上变成 Tile"算"它**。


## 2. Tensor 声明与方向标注

Kernel 签名中的 Tensor 类型标注：`pl.Tensor[[shape...], dtype, (可选)layout, (可选)direction]`。

```python
@pl.jit(auto_mutex=True)
def kernel(
    x: pl.Tensor[[pl.DYNAMIC, 128], pl.DT_FP16, pl.Input],
    out: pl.Tensor[[pl.DYNAMIC, 128], pl.DT_FP16, pl.Output],
):
    ...
```

- `pl.Input` / `pl.Output` 是方向标注，**只影响 Profiling 结果中的上报信息，不改变读写语义**；默认 Input；
- layout 标注（如 `pl.NZ`）只描述 GM 中已有数据的排布，不做格式转换。


## 3. make_tensor：从指针构造视图

`pl.make_tensor(ptr, shape, stride=None, dtype=None)` 从 Ptr 或已有 Tensor 创建**共享同一 GM 地址**的新视图，不申请内存、不复制数据。

```python
# 连续视图：省略 stride 时按行主序生成
tensor = pl.make_tensor(ptr, [8, 16])            # 等价 [8,16],[16,1]

# 显式 stride：行间不连续（pitched）
pitched = pl.make_tensor(ptr, [8, 16], [32, 1])  # 相邻行起始相隔 32 元素

# 转置视图：normal[i,j] 与 transposed[j,i] 指向相同地址
normal = pl.make_tensor(ptr, [8, 16], [16, 1])
transposed = pl.make_tensor(ptr, [16, 8], [1, 16])
```

【待 950 验证：以上为官方文档示例】

两个要点：stride 单位是**元素不是字节**；转置视图只改变"逻辑索引 → 地址"的映射，**不搬运数据**——真正的转置要靠搬运接口完成。

### 配合 TilingData 处理动态 shape

```python
@pl.jit(auto_mutex=True)
def dynamic_kernel(x: pl.Ptr[pl.DT_FP16], out: pl.Ptr[pl.DT_FP16], tiling: OpTiling):
    tensor_x = pl.make_tensor(x, [tiling.m, tiling.n])
    tensor_out = pl.make_tensor(out, [tiling.m, tiling.n])
    ...
```


## 4. 读取 Shape 与指针操作

- Kernel 内用 `x.shape[axis]` 读取维度（axis 为编译期整数，支持负索引），可参与地址计算、循环边界与多核切分；
- `pl.make_ptr(tensor)` 从 Tensor 提取底层指针；`pl.make_ptr(byte_ptr, dtype=...)` 重解释元素类型；
- 指针偏移用 `addptr`（按元素偏移），偏移后再 make_tensor 包装——常用于把一块 Workspace 划分为多个区域；
- **Tensor 不支持切片**（如 `tensor[0:32, :]`）：访问局部数据用 load/store 的坐标定位，或 addptr + make_tensor。


## 5. Tile：规格（TileType）先于实体

Tile 的创建分两步（详见 02.04~02.05）：先用 `TileType` 描述规格（shape/dtype/target_memory/layout 等），再用 `make_tile`（单块）或 `make_tile_group`（一组轮转）绑定片上地址。此处先建立概念：**TileType 只是"图纸"，不分配空间**。


## 6. Tensor 与 Tile 之间的搬运

Tensor 位于 GM，**不能直接作为 Tile 计算接口的操作数**——必须先搬入：

```python
with pl.section_vector():
    pl.load(input_tile, input_tensor, [row_off, col_off])   # GM -> 片上
    # ... Tile 计算 ...
    pl.store(output_tensor, output_tile, [row_off, col_off]) # 片上 -> GM
```

`load/store` 用**元素坐标**定位搬运起点；`load_tile/store_tile` 按 **Tile 网格坐标**定位数据块（大 Tensor 分块搬运时更方便）。


### 手算一个 pitched 视图的地址

假设 FP32 基址为 P，shape=[3,4]、stride=[8,1]。元素 `[2,3]` 的元素偏移为 `2×8+3=19`，字节地址为 `P+76`。底层存储至少要覆盖 20 个元素，虽然逻辑矩阵只含 12 个元素。Host 如果只申请 12 个连续元素，再构造该视图，会把“视图合法”误当成“底层内存足够”。

对连续 `[8,256]`，`[4,512]` 的新视图仍覆盖 2048 个元素，默认 stride 变成 [512,1]；数据并未搬移。`load(tile,tensor,[4,0])` 中 4 是源 Tensor 的行坐标；片上 `addrs=0x4000` 则是字节偏移。这两个数字属于不同坐标系，不应直接互相替换。

**自查**：将上述 pitched 视图改为 FP16，元素偏移仍是 19，字节偏移变为 38。


## 7. 课后练习

1. （填空）Tensor 描述 GM 数据视图但不申请内存，GM 内存由 ______ 侧的 ______ 分配。
2. （判断）`pl.make_tensor(ptr, [16, 8], [1, 16])` 会把数据在内存中转置。（对/错）
3. （实践）一个 [8, 256] FP32 的 Tensor，若要把它"看作" [4, 512] 处理，make_tensor 应该怎么写？数据总字节数变了吗？

In [ ]:
!cat ./answer/02.03_answer.txt

## 8. 本节小结

- 三类对象各司其职：**Tensor 看 GM、Tile 算片上、Ptr 做桥梁**；
- make_tensor 构造视图（stride 单位是元素，不搬运数据）；Tensor 不支持切片；
- Tensor → Tile 必须显式搬运（load/store 元素坐标、load_tile/store_tile 网格坐标）。